# Day 070 — Exercise 3: Generate Image

**What you'll build:** `generate_image(prompt, negative, generate_fn, width, height, steps, guidance_scale, seed)` — the core generation function that forwards all parameters to the mock or real pipeline.

**Why it matters:** All higher-level functions (`generate_variations`, `ImageGenerator.generate`) delegate to this one. Getting parameter forwarding right here means downstream functions automatically work correctly.

In [ ]:
from PIL import Image
_mock_gen = lambda prompt, **kw: Image.new('RGB', (kw.get('width', 512), kw.get('height', 512)), 'steelblue')


## Task

Implement `generate_image`:

1. If `generate_fn is not None`: return `generate_fn(prompt, negative=negative, width=width, height=height, steps=steps, guidance_scale=guidance_scale, seed=seed)`
2. Otherwise: use `diffusers.StableDiffusionPipeline` (lazy import — only needed for real generation)

The checks always use `_mock_gen`, so the diffusers path is not required to work in exercises.

## Your Implementation

In [ ]:
def generate_image(prompt: str, negative: str = '',
                   generate_fn=None,
                   width: int = 512, height: int = 512,
                   steps: int = 20, guidance_scale: float = 7.5,
                   seed: int = 42):
    """Generate an image from a text prompt.

    Args:
        prompt:         Positive text prompt
        negative:       Negative text prompt
        generate_fn:    callable(prompt, **kwargs) -> PIL.Image for testing
        width:          Output width in pixels
        height:         Output height in pixels
        steps:          Denoising steps
        guidance_scale: CFG scale
        seed:           Random seed
    Returns:
        PIL.Image.Image
    """
    raise NotImplementedError


In [ ]:
def generate_image(prompt: str, negative: str = '',
                   generate_fn=None,
                   width: int = 512, height: int = 512,
                   steps: int = 20, guidance_scale: float = 7.5,
                   seed: int = 42):
    if generate_fn is not None:
        return generate_fn(prompt, negative=negative, width=width,
                           height=height, steps=steps,
                           guidance_scale=guidance_scale, seed=seed)
    from diffusers import StableDiffusionPipeline
    import torch
    pipe = StableDiffusionPipeline.from_pretrained(
        'runwayml/stable-diffusion-v1-5',
        torch_dtype=torch.float32,
    )
    generator = torch.Generator().manual_seed(seed)
    result = pipe(prompt, negative_prompt=negative or None,
                  width=width, height=height,
                  num_inference_steps=steps,
                  guidance_scale=guidance_scale,
                  generator=generator)
    return result.images[0]


## Automated checks

In [ ]:
score, total = 0, 5
try:
    img = generate_image('a cat on a moon', generate_fn=_mock_gen)
    assert isinstance(img, Image.Image), f"Expected PIL Image, got {type(img)}"
    score += 1; print("\u2705 returns PIL Image")

    assert img.size == (512, 512), f"Expected (512,512), got {img.size}"
    assert img.mode == 'RGB'
    score += 1; print("\u2705 default size (512, 512) RGB")

    img2 = generate_image('a dog', generate_fn=_mock_gen, width=256, height=128)
    assert img2.size == (256, 128), f"Expected (256,128), got {img2.size}"
    score += 1; print("\u2705 width/height propagated to generate_fn")

    # Verify generate_fn receives all params
    captured = {}
    def _capture(prompt, **kw):
        captured.update({'prompt': prompt, **kw})
        return Image.new('RGB', (kw.get('width', 64), kw.get('height', 64)), 'red')

    generate_image('sky', negative='clouds', generate_fn=_capture,
                   steps=30, guidance_scale=9.0, seed=99)
    assert captured.get('steps') == 30
    assert abs(captured.get('guidance_scale', 0) - 9.0) < 0.001
    assert captured.get('seed') == 99
    score += 1; print("\u2705 all params forwarded to generate_fn")

    assert captured.get('negative') == 'clouds'
    score += 1; print("\u2705 negative prompt forwarded")

except Exception as e:
    print(f"\u274c {e}")

print(f"\n{score}/{total} checks passed")
if score == total:
    print("\U0001f389 Exercise complete!")


## Solution

<details><summary>Reveal</summary>

```python
def generate_image(prompt: str, negative: str = '',
                   generate_fn=None,
                   width: int = 512, height: int = 512,
                   steps: int = 20, guidance_scale: float = 7.5,
                   seed: int = 42):
    if generate_fn is not None:
        return generate_fn(prompt, negative=negative, width=width,
                           height=height, steps=steps,
                           guidance_scale=guidance_scale, seed=seed)
    from diffusers import StableDiffusionPipeline
    import torch
    pipe = StableDiffusionPipeline.from_pretrained(
        'runwayml/stable-diffusion-v1-5',
        torch_dtype=torch.float32,
    )
    generator = torch.Generator().manual_seed(seed)
    result = pipe(prompt, negative_prompt=negative or None,
                  width=width, height=height,
                  num_inference_steps=steps,
                  guidance_scale=guidance_scale,
                  generator=generator)
    return result.images[0]
```

**Why lazy import diffusers?** Diffusers and torch are multi-gigabyte packages. By importing them only inside the `else` branch, the module loads cleanly in any environment. Students who have only PIL installed can still use all exercises via mock injection.

</details>